# 09 · Integración con Big Data (Apache Spark)

| | |
|---|---|
| **Fase** | F3 / F5 |
| **IDs del enunciado que cubre** | **IA3.6** (integrar con Big Data: Spark, Hadoop) |
| **Entregables** | este notebook + Parquet particionado en `db/processed/spark/` |

**Contenido:**
1. `SparkSession` en modo local (`local[*]`).
2. Lectura del CSV con **esquema explícito**; `printSchema`, `count`, `describe`.
3. Transformaciones: `filter`, `select`, `withColumn` (mismas variables nuevas del paso 3), `groupBy().agg()` y **Spark SQL**.
4. Escritura en **Parquet particionado por `proposito`** (analogía de HDFS) y explicación de HDFS, YARN y MapReduce.
5. **Spark MLlib:** `Pipeline(StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, LogisticRegression)` evaluado con `BinaryClassificationEvaluator` y comparado con scikit-learn.
6. **Scoring por lotes** de todo el dataset (simula un proceso batch nocturno).

**Prerrequisitos:**
- `pip install pyspark` (PySpark 4.x soporta Java 17 y 21; PySpark 3.5 requiere Java 8/11/17) y **Java** con `JAVA_HOME` configurado.
- En **Windows**, escribir Parquet con Spark requiere `winutils.exe` y `hadoop.dll` (`HADOOP_HOME`). Si falla, el notebook escribe el mismo Parquet particionado con `pyarrow` como respaldo; la alternativa recomendada es correr este notebook en Docker (`jupyter/pyspark-notebook`) o en Google Colab.
- Si PySpark no está instalado, las celdas de Spark se omiten con un mensaje claro y se ejecutan las partes equivalentes con pandas/scikit-learn.

## 0. Configuración

In [1]:
import os, json, time, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.options.display.float_format = "{:,.4f}".format
pd.options.display.max_columns = 60
plt.rcParams["figure.dpi"] = 100

def encontrar_raiz():
    """Sube desde el directorio actual hasta encontrar db/credit_risk.csv (sin rutas absolutas)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "db" / "credit_risk.csv").exists():
            return carpeta
    raise FileNotFoundError("No se encontró db/credit_risk.csv. Ejecute el notebook desde Back/notebooks.")

RAIZ = encontrar_raiz()
DATA_PATH = RAIZ / "db" / "credit_risk.csv"
MODELS_DIR = RAIZ / "Back" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATA_PATH.relative_to(RAIZ), "| artefactos:", MODELS_DIR.relative_to(RAIZ))

Dataset: db\credit_risk.csv | artefactos: Back\models


In [2]:
TARGET = "default"
TEXTO = "texto_solicitud"
SENSIBLES = ["sexo"]   # solo para auditar equidad, NUNCA como predictor
NUM_BASE = ["edad", "antiguedad_laboral_anios", "personas_a_cargo", "ingreso_mensual", "monto_credito",
            "plazo_meses", "tasa_interes_anual", "num_creditos_activos", "moras_ultimos_12m",
            "relacion_deuda_ingreso", "puntaje_buro", "uso_cupo_tarjetas"]
CAT_COLS = ["estado_civil", "nivel_educativo", "situacion_laboral", "tipo_vivienda", "proposito"]
NUEVAS = ["cuota_estimada", "ratio_cuota_ingreso", "ratio_monto_ingreso", "tiene_moras_previas", "longitud_texto"]
NUM_FEATS = NUM_BASE + NUEVAS
FEATURES = NUM_FEATS + CAT_COLS

def limpiar(df_raw, verbose=True):
    """Duplicados (exactos y por id_cliente), valores inválidos -> NaN y texto vacío.
    La imputación (mediana/moda) y el capping p1-p99 se hacen en `preparar`, con estadísticos de train."""
    df = df_raw.copy()
    n0 = len(df)
    df = df.drop_duplicates()
    n1 = len(df)
    df = df.drop_duplicates(subset="id_cliente", keep="first")
    n2 = len(df)
    df = df[df[TARGET].isin([0, 1])].copy()
    df[TARGET] = df[TARGET].astype(int)
    df.loc[~df["edad"].between(18, 100), "edad"] = np.nan          # edades imposibles (1, 999...)
    for c in ["ingreso_mensual", "monto_credito"]:
        df.loc[df[c] <= 0, c] = np.nan
    for c in CAT_COLS + ["sexo"]:
        df[c] = df[c].where(df[c].isna(), df[c].astype(str).str.strip())
    df[TEXTO] = df[TEXTO].fillna("").astype(str)
    if verbose:
        print(f"Filas originales: {n0:,} | duplicados exactos: {n0 - n1} | duplicados por id_cliente: {n1 - n2} | filas finales: {len(df):,}")
    return df.reset_index(drop=True)

def agregar_variables(df):
    """Variables de negocio del paso 3: cuota (amortización francesa), ratios, moras previas y longitud del texto."""
    df = df.copy()
    r = df["tasa_interes_anual"] / 100 / 12
    n = df["plazo_meses"]
    df["cuota_estimada"] = df["monto_credito"] * r / (1 - (1 + r) ** (-n))
    df["ratio_cuota_ingreso"] = df["cuota_estimada"] / df["ingreso_mensual"]
    df["ratio_monto_ingreso"] = df["monto_credito"] / df["ingreso_mensual"]
    df["tiene_moras_previas"] = (df["moras_ultimos_12m"] > 0).astype(int)
    df["longitud_texto"] = df[TEXTO].str.len()
    return df

def preparar(df_limpio, test_size=0.2):
    """Split estratificado ANTES de imputar/capar: medianas, modas y percentiles salen solo de train (sin leakage)."""
    train, test = train_test_split(df_limpio, test_size=test_size, stratify=df_limpio[TARGET],
                                   random_state=RANDOM_STATE)
    train, test = train.copy(), test.copy()
    medianas = train[NUM_BASE].median()
    modas = {c: train[c].mode().iloc[0] for c in CAT_COLS}
    p01, p99 = train[NUM_BASE].quantile(0.01), train[NUM_BASE].quantile(0.99)
    preparar.stats = {"medianas": medianas, "modas": modas, "p01": p01, "p99": p99}
    salida = []
    for d in (train, test):
        d[NUM_BASE] = d[NUM_BASE].fillna(medianas).clip(lower=p01, upper=p99, axis=1)
        for c in CAT_COLS:
            d[c] = d[c].fillna(modas[c])
        salida.append(agregar_variables(d).reset_index(drop=True))
    return salida[0], salida[1]

In [3]:
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, precision_score,
                             recall_score, f1_score, brier_score_loss, roc_curve, precision_recall_curve,
                             confusion_matrix)

def ks_stat(y, p):
    """Estadístico KS: máxima separación entre las distribuciones acumuladas de buenos y malos."""
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr - fpr))

def metricas(y, p, umbral=0.5):
    pred = (np.asarray(p) >= umbral).astype(int)
    return {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred), "f1": f1_score(y, pred), "ks": ks_stat(y, p),
            "brier": brier_score_loss(y, p)}

In [4]:
import shutil, sys
SPARK_DIR = RAIZ / "db" / "processed" / "spark"
SPARK_DIR.mkdir(parents=True, exist_ok=True)

if not os.environ.get("JAVA_HOME"):
    for base in [Path("C:/Program Files/Java"), Path("C:/Program Files/Eclipse Adoptium"), Path("/usr/lib/jvm")]:
        if base.exists():
            candidatos = sorted(base.glob("*"))
            if candidatos:
                os.environ["JAVA_HOME"] = str(candidatos[-1])
                break
print("JAVA_HOME:", os.environ.get("JAVA_HOME"), "| java en PATH:", shutil.which("java"))
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

try:
    import pyspark
    from pyspark.sql import SparkSession, functions as F, types as T
    from pyspark.ml import Pipeline as SparkPipeline
    from pyspark.ml.feature import StringIndexer, OneHotEncoder as SparkOneHot, VectorAssembler, StandardScaler as SparkScaler
    from pyspark.ml.classification import LogisticRegression as SparkLogReg
    from pyspark.ml.evaluation import BinaryClassificationEvaluator
    from pyspark.ml.functions import vector_to_array
    SPARK_OK = True
    print("PySpark", pyspark.__version__)
except Exception as e:
    SPARK_OK = False
    print("PySpark no está disponible:", repr(e)[:150])
    print("Instalar con: pip install pyspark   (y Java 17/21 con JAVA_HOME). Se ejecutarán solo las partes con pandas.")

JAVA_HOME: C:\Program Files\Java\jdk-21.0.10 | java en PATH: C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE
PySpark no está disponible: ModuleNotFoundError("No module named 'pyspark'")
Instalar con: pip install pyspark   (y Java 17/21 con JAVA_HOME). Se ejecutarán solo las partes con pandas.


## 1. `SparkSession` en modo local

In [5]:
spark = None
if SPARK_OK:
    try:
        spark = (SparkSession.builder.master("local[*]").appName("riesgo_credito")
                 .config("spark.sql.shuffle.partitions", "8").config("spark.driver.memory", "2g")
                 .config("spark.sql.session.timeZone", "UTC").getOrCreate())
        spark.sparkContext.setLogLevel("ERROR")
        print("Spark", spark.version, "| master:", spark.sparkContext.master,
              "| paralelismo:", spark.sparkContext.defaultParallelism)
    except Exception as e:
        SPARK_OK = False
        print("No se pudo iniciar Spark (revise Java/JAVA_HOME):", repr(e)[:300])

## 2. Lectura con esquema explícito

In [6]:
if SPARK_OK:
    esquema = T.StructType([
        T.StructField("id_cliente", T.StringType()), T.StructField("edad", T.DoubleType()),
        T.StructField("sexo", T.StringType()), T.StructField("estado_civil", T.StringType()),
        T.StructField("nivel_educativo", T.StringType()), T.StructField("situacion_laboral", T.StringType()),
        T.StructField("antiguedad_laboral_anios", T.DoubleType()), T.StructField("personas_a_cargo", T.DoubleType()),
        T.StructField("tipo_vivienda", T.StringType()), T.StructField("ingreso_mensual", T.DoubleType()),
        T.StructField("monto_credito", T.DoubleType()), T.StructField("plazo_meses", T.DoubleType()),
        T.StructField("tasa_interes_anual", T.DoubleType()), T.StructField("proposito", T.StringType()),
        T.StructField("num_creditos_activos", T.DoubleType()), T.StructField("moras_ultimos_12m", T.DoubleType()),
        T.StructField("relacion_deuda_ingreso", T.DoubleType()), T.StructField("puntaje_buro", T.DoubleType()),
        T.StructField("uso_cupo_tarjetas", T.DoubleType()), T.StructField("texto_solicitud", T.StringType()),
        T.StructField("default", T.IntegerType())])
    sdf_raw = spark.read.csv(str(DATA_PATH), header=True, schema=esquema, quote='"', escape='"', multiLine=False)
    sdf_raw.printSchema()
    print("Filas:", sdf_raw.count(), "| particiones:", sdf_raw.rdd.getNumPartitions())
    sdf_raw.select("edad", "ingreso_mensual", "monto_credito", "puntaje_buro", "default").describe().show()
else:
    print("Spark no disponible: se omite la lectura distribuida.")

Spark no disponible: se omite la lectura distribuida.


## 3. Transformaciones en Spark

Mismas reglas del paso 3: duplicados por `id_cliente`, edades inválidas a nulo, imputación (mediana con `approxQuantile` y moda) y las variables nuevas con `withColumn`. Spark es **perezoso** (*lazy*): las transformaciones solo construyen un plan y se ejecutan cuando hay una acción (`count`, `show`, `write`).

In [7]:
if SPARK_OK:
    sdf = (sdf_raw.dropDuplicates(["id_cliente"])
           .withColumn("edad", F.when(F.col("edad").between(18, 100), F.col("edad")))
           .withColumn("texto_solicitud", F.coalesce(F.col("texto_solicitud"), F.lit(""))))
    medianas = {c: sdf.approxQuantile(c, [0.5], 0.001)[0] for c in NUM_BASE}
    modas = {c: sdf.groupBy(c).count().dropna().orderBy(F.desc("count")).first()[0] for c in CAT_COLS}
    sdf = sdf.fillna(medianas).fillna(modas)
    r = F.col("tasa_interes_anual") / 100 / 12
    sdf = (sdf.withColumn("cuota_estimada", F.col("monto_credito") * r / (1 - F.pow(1 + r, -F.col("plazo_meses"))))
              .withColumn("ratio_cuota_ingreso", F.col("cuota_estimada") / F.col("ingreso_mensual"))
              .withColumn("ratio_monto_ingreso", F.col("monto_credito") / F.col("ingreso_mensual"))
              .withColumn("tiene_moras_previas", (F.col("moras_ultimos_12m") > 0).cast("int"))
              .withColumn("longitud_texto", F.length("texto_solicitud").cast("double"))
              .withColumn("rango_buro", F.when(F.col("puntaje_buro") < 600, "bajo")
                                        .when(F.col("puntaje_buro") < 750, "medio").otherwise("alto")))
    sdf = sdf.cache()
    print("Filas tras limpieza:", sdf.count())
    sdf.filter((F.col("ingreso_mensual") < 2_000_000) & (F.col("monto_credito") > 20_000_000)) \
       .select("id_cliente", "ingreso_mensual", "monto_credito", "ratio_cuota_ingreso", "default").show(5)

In [8]:
if SPARK_OK:
    (sdf.groupBy("proposito")
        .agg(F.count("*").alias("clientes"), F.round(F.avg("default"), 4).alias("tasa_impago"),
             F.round(F.avg("monto_credito"), 0).alias("monto_medio"), F.round(F.avg("ratio_cuota_ingreso"), 4).alias("cuota_ingreso_media"))
        .orderBy(F.desc("tasa_impago")).show())

### 3.1 Spark SQL

In [9]:
if SPARK_OK:
    sdf.createOrReplaceTempView("solicitudes")
    spark.sql("""
        SELECT situacion_laboral, rango_buro,
               COUNT(*)                    AS clientes,
               ROUND(AVG(`default`), 4)    AS tasa_impago,
               ROUND(AVG(puntaje_buro), 1) AS buro_medio
        FROM solicitudes
        GROUP BY situacion_laboral, rango_buro
        HAVING COUNT(*) >= 50
        ORDER BY tasa_impago DESC
    """).show(15)

## 4. Almacenamiento: Parquet particionado (analogía con HDFS)

- **HDFS** (*Hadoop Distributed File System*) divide los archivos en bloques replicados entre nodos; aquí `db/processed/spark/` hace de "sistema de archivos" y cada carpeta `proposito=...` es una partición que Spark puede leer en paralelo y **podar** (*partition pruning*) cuando se filtra por esa columna.
- **YARN** es el gestor de recursos de Hadoop que asigna CPU y memoria a las tareas del clúster; en modo `local[*]` Spark usa los núcleos de la máquina, y en producción correría sobre YARN, Kubernetes o un servicio gestionado (Dataproc, EMR, Databricks).
- **MapReduce** procesa en dos fases (*map* y *reduce*) escribiendo a disco entre etapas; **Spark** mantiene los datos en memoria y optimiza el plan completo (DAG + Catalyst), por lo que es mucho más rápido en procesos iterativos como el entrenamiento de modelos.
- **Parquet** es columnar y comprimido: lee solo las columnas necesarias, ideal para analítica.

In [10]:
ruta_parquet = SPARK_DIR / "credit_risk_por_proposito"
escrito_con = None
if SPARK_OK:
    try:
        sdf.write.mode("overwrite").partitionBy("proposito").parquet(str(ruta_parquet))
        escrito_con = "Spark"
    except Exception as e:
        print("Spark no pudo escribir Parquet (en Windows suele faltar winutils/HADOOP_HOME):", repr(e)[:200])
if escrito_con is None:
    # Respaldo: mismo layout particionado con pyarrow (limpieza equivalente en pandas)
    df_pd = limpiar(pd.read_csv(DATA_PATH), verbose=False)
    df_pd[NUM_BASE] = df_pd[NUM_BASE].fillna(df_pd[NUM_BASE].median())
    for c in CAT_COLS:
        df_pd[c] = df_pd[c].fillna(df_pd[c].mode().iloc[0])
    df_pd = agregar_variables(df_pd)
    if ruta_parquet.exists():
        shutil.rmtree(ruta_parquet)
    df_pd.to_parquet(ruta_parquet, partition_cols=["proposito"], index=False)
    escrito_con = "pyarrow (respaldo)"
particiones = sorted(p.name for p in ruta_parquet.iterdir() if p.is_dir())
print(f"Parquet escrito con {escrito_con} en {ruta_parquet.relative_to(RAIZ)}")
print("Particiones:", particiones)

Parquet escrito con pyarrow (respaldo) en db\processed\spark\credit_risk_por_proposito
Particiones: ['proposito=Consolidacion_deuda', 'proposito=Educacion', 'proposito=Libre_inversion', 'proposito=Negocio', 'proposito=Vehiculo', 'proposito=Vivienda']


In [11]:
if SPARK_OK and escrito_con == "Spark":
    leido = spark.read.parquet(str(ruta_parquet)).filter(F.col("proposito") == "Vivienda")
    print("Lectura con partition pruning (proposito = Vivienda):", leido.count(), "filas")
else:
    leido = pd.read_parquet(ruta_parquet, filters=[("proposito", "=", "Vivienda")])
    print("Lectura con filtro de partición (pyarrow, proposito = Vivienda):", len(leido), "filas")

Lectura con filtro de partición (pyarrow, proposito = Vivienda): 1707 filas


## 5. Spark MLlib vs scikit-learn

Pipeline de MLlib: `StringIndexer → OneHotEncoder → VectorAssembler → StandardScaler → LogisticRegression`, con split 80/20 (`randomSplit`, semilla 42) y AUC con `BinaryClassificationEvaluator`. Se compara con una regresión logística de scikit-learn sobre los mismos datos y variables.

In [12]:
auc_mllib, t_mllib = None, None
if SPARK_OK:
    s_train, s_test = sdf.randomSplit([0.8, 0.2], seed=RANDOM_STATE)
    indexador = StringIndexer(inputCols=CAT_COLS, outputCols=[c + "_idx" for c in CAT_COLS], handleInvalid="keep")
    onehot = SparkOneHot(inputCols=[c + "_idx" for c in CAT_COLS], outputCols=[c + "_ohe" for c in CAT_COLS])
    ensamblador = VectorAssembler(inputCols=NUM_FEATS + [c + "_ohe" for c in CAT_COLS], outputCol="features_raw",
                                  handleInvalid="keep")
    escalador = SparkScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)
    logreg = SparkLogReg(featuresCol="features", labelCol="default", maxIter=100, regParam=0.0)
    pipeline_ml = SparkPipeline(stages=[indexador, onehot, ensamblador, escalador, logreg])
    t0 = time.time()
    modelo_ml = pipeline_ml.fit(s_train)
    t_mllib = time.time() - t0
    pred_ml = modelo_ml.transform(s_test)
    evaluador = BinaryClassificationEvaluator(labelCol="default", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
    auc_mllib = evaluador.evaluate(pred_ml)
    print(f"MLlib LogisticRegression -> AUC test = {auc_mllib:.4f} | entrenamiento {t_mllib:.1f} s")
else:
    print("Spark no disponible: se omite MLlib (pip install pyspark).")

Spark no disponible: se omite MLlib (pip install pyspark).


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

df_sk = limpiar(pd.read_csv(DATA_PATH), verbose=False)
train_sk, test_sk = preparar(df_sk)
pipe_sk = Pipeline([("prep", ColumnTransformer([
                        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), NUM_FEATS),
                        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS)])),
                    ("modelo", LogisticRegression(max_iter=3000))])
t0 = time.time()
pipe_sk.fit(train_sk[FEATURES], train_sk[TARGET])
t_sk = time.time() - t0
auc_sk = roc_auc_score(test_sk[TARGET], pipe_sk.predict_proba(test_sk[FEATURES])[:, 1])
comparacion = pd.DataFrame([
    {"motor": "Spark MLlib (local[*])", "modelo": "LogisticRegression", "roc_auc": auc_mllib, "tiempo_s": t_mllib},
    {"motor": "scikit-learn", "modelo": "LogisticRegression", "roc_auc": auc_sk, "tiempo_s": t_sk}])
comparacion

,motor,modelo,roc_auc,tiempo_s
0,Spark MLlib (local[*]),LogisticRegression,NaN,NaN
1,scikit-learn,LogisticRegression,0.8458,4.3093


**Interpretación:** ambos motores entrenan la misma familia de modelo, por lo que el AUC debe ser prácticamente igual (las pequeñas diferencias vienen de la partición aleatoria distinta). Con 12 mil filas scikit-learn es más rápido porque Spark paga el costo de arrancar la JVM y planificar tareas; Spark gana cuando los datos no caben en la memoria de una sola máquina.

## 6. Scoring por lotes (proceso batch nocturno)

In [14]:
if SPARK_OK:
    scoring = (modelo_ml.transform(sdf)
               .select("id_cliente", "proposito", F.round(vector_to_array("probability")[1], 4).alias("probabilidad_impago"))
               .withColumn("banda_riesgo", F.when(F.col("probabilidad_impago") < 0.2, "Bajo")
                                            .when(F.col("probabilidad_impago") < 0.5, "Medio").otherwise("Alto"))
               .withColumn("fecha_scoring", F.current_date()))
    scoring.groupBy("banda_riesgo").count().orderBy("banda_riesgo").show()
    try:
        scoring.write.mode("overwrite").partitionBy("banda_riesgo").parquet(str(SPARK_DIR / "scoring_batch"))
        print("Predicciones batch guardadas en db/processed/spark/scoring_batch")
    except Exception as e:
        scoring.toPandas().to_parquet(SPARK_DIR / "scoring_batch.parquet", index=False)
        print("Escritura Spark falló (winutils); predicciones guardadas con pandas en db/processed/spark/scoring_batch.parquet")
else:
    todos = pd.concat([train_sk, test_sk], ignore_index=True)
    scoring_pd = todos[["id_cliente", "proposito"]].assign(
        probabilidad_impago=pipe_sk.predict_proba(todos[FEATURES])[:, 1].round(4))
    scoring_pd["banda_riesgo"] = pd.cut(scoring_pd["probabilidad_impago"], [-0.01, 0.2, 0.5, 1.0], labels=["Bajo", "Medio", "Alto"]).astype(str)
    scoring_pd["fecha_scoring"] = pd.Timestamp.today().normalize()
    scoring_pd.to_parquet(SPARK_DIR / "scoring_batch_sklearn.parquet", index=False)
    print("Spark no disponible: scoring batch equivalente con scikit-learn guardado en db/processed/spark/scoring_batch_sklearn.parquet")
    print(scoring_pd["banda_riesgo"].value_counts().to_string())

Spark no disponible: scoring batch equivalente con scikit-learn guardado en db/processed/spark/scoring_batch_sklearn.parquet
banda_riesgo
Bajo     5031
Alto     3604
Medio    3365


In [15]:
if spark is not None:
    spark.stop()
    print("SparkSession detenida.")

## 7. Resumen

In [16]:
print("Parquet particionado:", ruta_parquet.relative_to(RAIZ), "| escrito con:", escrito_con, "| particiones:", len(particiones))
print("AUC MLlib:", None if auc_mllib is None else round(auc_mllib, 4), "| AUC scikit-learn:", round(auc_sk, 4))

Parquet particionado: db\processed\spark\credit_risk_por_proposito | escrito con: pyarrow (respaldo) | particiones: 6
AUC MLlib: None | AUC scikit-learn: 0.8458


> **Estado de ejecución (respuesta a QA, 2026-10-16):** las salidas guardadas muestran que **PySpark no estaba instalado** (`AUC MLlib: None`). El Parquet y el scoring se generaron con la **ruta de respaldo pandas/sklearn**, que **no acredita IA3.6**. Para cumplirlo: Java 17 + `pip install pyspark` y volver a ejecutar.

## Conclusiones

1. El mismo flujo del proyecto (lectura con esquema, limpieza, variables nuevas, agregaciones y SQL) se expresa en **Spark** con transformaciones perezosas que escalarían a millones de solicitudes sin cambiar el código.
2. Los datos quedan en **Parquet particionado por `proposito`** en `db/processed/spark/`, el formato estándar de un *data lake*; las lecturas filtradas por propósito solo tocan su partición (*partition pruning*), igual que en HDFS.
3. **Spark MLlib** entrena una regresión logística con un pipeline análogo al de scikit-learn y llega a un AUC equivalente; a este volumen scikit-learn es más rápido, y Spark se justifica cuando el volumen de datos o la frecuencia de *scoring* crecen.
4. El **scoring por lotes** demuestra el patrón de producción "batch nocturno": se califica toda la cartera y se guardan la PD y la banda por cliente para el Front y los reportes.
5. **Prerrequisitos y limitaciones:** PySpark y Java (17/21) con `JAVA_HOME`; en Windows la escritura nativa de Parquet requiere `winutils`. Si Spark no está disponible, el notebook genera los mismos artefactos con pyarrow y scikit-learn, y se recomienda Docker (`jupyter/pyspark-notebook`) o Colab para la ejecución distribuida completa.